# NeuroTutorSim Phase II: TRIBE v2 prediction dataset

Deliverable **D3** of the project brief (§6.1-6.5, §10.1): Meta's TRIBE v2 encoding model is run on the Phase I
corpus (30 units x 3 instructional conditions = 90 stimuli) and this notebook writes, for every stimulus,

* the complete time-resolved vertex predictions (`tribe_vertex`, brief §4.2 / §6.3),
* the parcel summaries `tribe_parcel.parquet` (eq. 6) and the network time courses (eq. 7),
* the immediate neural metrics of §6.5 (eq. 8-9) per parcel and per network,
* the quality checks of §10.1 (checkpoint hash, official example, determinism, shuffled-text controls).

Condition contrasts (§6.6) and representational similarity (§6.7) need no GPU: they are computed offline from
`tribe_metrics.parquet` with `neurotutorsim.tribe.fixed_effects` and `neurotutorsim.tribe.rsa`.

**Interpretation rule (brief §1, §15).** Every number here is a *predicted cortical response of an average adult
reader* to a stimulus. It is not a measurement, not learning, and not plasticity; Phases IV-V consume the `auc` per
`(unit_id, condition, parcel)` through eq. 28 and nothing else.

**How to run (Google Colab).**
1. `Runtime > Change runtime type > GPU` (a T4 works; an L4/A100 is ~3x faster).
2. Colab secrets (key icon in the left bar): `HF_TOKEN`, a Hugging Face token whose account has accepted the
   `meta-llama/Llama-3.2-3B` licence (TRIBE's text encoder is gated), and `GITHUB_TOKEN` if the corpus is fetched from
   the private GitHub repository (or use `CORPUS_SOURCE = "drive"` / `"upload"`).
3. Edit the configuration cell, then `Runtime > Run all`. The first run installs TRIBE v2 and asks you to **restart the
   runtime once**; run all again afterwards.
4. Outputs land in `DRIVE_OUTPUT_DIR/<RUN_TAG>/` on Google Drive (synced to your computer if you use Drive for desktop)
   and a zip of the summary tables is offered for download at the end.

**Budget.** Text features for the 90 stimuli are ~53k word-in-context forward passes of Llama-3.2-3B (about 40-60 min
on a T4, ~15 min on an A100); the encoder itself takes seconds per stimulus. Everything is cached and the run is
resumable: rerunning with the same `RUN_TAG` skips every stimulus whose predictions are already on disk and never
regenerates an existing item (brief §4.3).

In [ ]:
# ---- Configuration: edit here; nothing below needs changes ------------------------------------------------------
RUN_TAG = "tribe_main"            # output folder; one tag per run design (change it rather than mixing designs)
DRY_RUN = False                   # True = no model at all: deterministic FAKE predictions to smoke-test the pipeline.
                                  # Outputs are quarantined under DRYRUN_<RUN_TAG> and flagged in run_metadata.json.

# Where the corpus comes from
CORPUS_SOURCE = "github"          # "github" (private repo -> GITHUB_TOKEN secret) | "drive" (repo copy on Google Drive)
                                  # | "upload" (zip of the repo chosen in the browser) | "local" (running inside the repo)
GITHUB_REPO = "MatteoGuardamagna4/neurai"
GITHUB_REF = "main"               # branch, tag or commit; the commit actually used is written to run_metadata.json
DRIVE_REPO_DIR = "/content/drive/MyDrive/neurotutorsim/neurai"   # CORPUS_SOURCE == "drive"

# Where outputs go
SAVE_TO_DRIVE = True              # Colab: mount Google Drive and write there (else the runtime's local disk)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/neurotutorsim/tribe"
LOCAL_OUTPUT_DIR = ""             # non-Colab override; default is <repo>/data/processed/tribe (gitignored)
DOWNLOAD_ZIP = True               # offer a zip of the summary tables (not the vertex files) for download at the end
ZIP_INCLUDES_VERTEX = False       # ~1.6 GB per reading speed if True

# Model and timing
TRIBE_REPO = "https://github.com/facebookresearch/tribev2.git"
TRIBE_COMMIT = "af58661791a351a448a489042a28f6c37e1c14b7"   # main on 2026-06-23, pinned
TRIBE_HF_REPO = "facebook/tribev2"
TRIBE_CKPT_SHA256 = "9c79ffff6b642b7b0c71d558c935fb3fa33f2788bfb509feead94fafbba2f321"   # best.ckpt on the Hub
TRIBE_CKPT_BYTES = 708856138
READING_SPEEDS = [220, 180, 260]  # words per minute (brief §6.2): main specification first, then robustness
TEXT_PRECISION = "auto"           # "fp32" | "fp16" | "auto" (fp16 when the GPU has < 20 GB); recorded in the metadata
GROUP_SIZE = 10                   # stimuli per model.predict() call (each call loads the text encoder once)

# Parcellation (§6.4): Schaefer 2018 7-network parcels on fsaverage5, from the authors' repository at a pinned commit
SCHAEFER_PARCELS = 200            # 200 or 400
SCHAEFER_COMMIT = "d1454a611f7de10a3b36665e6fbb3fb6c770d140"

# Validation (§10.1)
RUN_OFFICIAL_DEMO = True          # reproduce the official text example before processing the corpus (decision gate 17)
RUN_CONTROLS = True               # determinism check and sentence- / word-shuffled stimuli
N_CONTROL_UNITS = 3               # units (x 3 conditions) that get shuffled controls
SEED = 20260909                   # the project's master seed (config/default.yaml)

## 1. Environment (brief §6.1 items 11-15)

TRIBE v2 is installed from the official repository at a pinned commit together with its exact dependency pins
(`torch<2.7`, `numpy==2.2.6`, ...). Colab ships newer versions, so the first run replaces them and needs a runtime
restart; the cell is skipped once `tribev2` imports.

In [ ]:
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not DRY_RUN and importlib.util.find_spec("tribev2") is None:
    print("Installing TRIBE v2 at", TRIBE_COMMIT[:12], "with its pinned dependencies (several minutes) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"tribev2 @ git+{TRIBE_REPO}@{TRIBE_COMMIT}",
                    "accelerate", "uv", "nibabel", "nilearn"], check=True)
    raise SystemExit("Installed. Restart the runtime now (Runtime > Restart session) and run all cells again.")
print("TRIBE v2 import:", "skipped (DRY_RUN)" if DRY_RUN else "ok")

In [ ]:
import hashlib, importlib.metadata, json, os, platform, shutil, tempfile, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

N_VERTICES = 20484  # fsaverage5, left then right hemisphere


def secret(name):
    """Colab secret if available, else the environment variable of the same name."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return os.environ.get(name, "")


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def version_of(package):
    try:
        return importlib.metadata.version(package)
    except Exception:
        return None


for name in ("HF_TOKEN", "GITHUB_TOKEN"):
    if secret(name):
        os.environ[name] = secret(name)

RUN = {
    "run_tag": RUN_TAG, "dry_run": DRY_RUN, "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB,
    "packages": {p: version_of(p) for p in ("numpy", "pandas", "pyarrow", "torch", "transformers", "neuralset",
                                             "neuraltrain", "exca", "nilearn", "nibabel", "huggingface_hub")},
    "reading_speeds_wpm": READING_SPEEDS, "seed": SEED, "text_precision_requested": TEXT_PRECISION,
    "group_size": GROUP_SIZE,
}
if not DRY_RUN:
    import torch
    RUN["torch"] = torch.__version__
    RUN["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        RUN["gpu"] = {"name": props.name, "memory_gb": round(props.total_memory / 1e9, 1)}
    else:
        print("WARNING: no GPU visible; TRIBE will run on CPU and take many hours.")
    RUN["tribev2_commit"] = TRIBE_COMMIT
    RUN["hf_token_present"] = bool(os.environ.get("HF_TOKEN"))
    if not RUN["hf_token_present"]:
        print("WARNING: no HF_TOKEN. The gated Llama-3.2-3B text encoder will fail to download; add the secret and rerun.")
print(json.dumps({k: v for k, v in RUN.items() if k != "packages"}, indent=1, default=str))

## 2. The corpus (Phase I output; decision gate 16)

The stimuli are the 90 markdown files under `stimuli/<condition>/<unit_id>.md`, validated by
`neurotutorsim.corpus` exactly as in the simulation repository (answers recomputed, sections checked, answer
leakage checked, duration caliper within 10 % of the traditional anchor). The join key with Phase III is
`(unit_id, condition)`; `stimulus_id = <unit_id>_<condition>` names the TRIBE timeline.

In [ ]:
def find_repo(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / "pyproject.toml").exists() and "neurotutorsim" in (p / "pyproject.toml").read_text(encoding="utf-8"):
            return p
    raise FileNotFoundError(f"no neurotutorsim repository at or above {start}")


def run_git(args, cwd=None):
    token = os.environ.get("GITHUB_TOKEN", "")
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git failed: " + (result.stderr or result.stdout).replace(token or "\0", "<token>"))
    return result.stdout.strip()


if CORPUS_SOURCE == "local":
    REPO_DIR = find_repo(Path.cwd())
elif CORPUS_SOURCE == "github":
    REPO_DIR = Path("/content/neurai") if IN_COLAB else Path.cwd() / "neurai"
    token = os.environ.get("GITHUB_TOKEN", "")
    url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not (REPO_DIR / ".git").exists():
        run_git(["clone", "--quiet", url, str(REPO_DIR)])
    run_git(["fetch", "--quiet", url, GITHUB_REF], cwd=REPO_DIR)
    run_git(["checkout", "--quiet", "FETCH_HEAD"], cwd=REPO_DIR)
elif CORPUS_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = find_repo(DRIVE_REPO_DIR)
elif CORPUS_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    target = Path("/content/neurai_upload")
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
        with zipfile.ZipFile(name) as z:
            z.extractall(target)
    REPO_DIR = next(p.parent for p in target.rglob("pyproject.toml"))
else:
    raise ValueError(f"unknown CORPUS_SOURCE {CORPUS_SOURCE!r}")

RUN["corpus"] = {"source": CORPUS_SOURCE, "repo_dir": str(REPO_DIR),
                 "commit": run_git(["rev-parse", "HEAD"], cwd=REPO_DIR) if (REPO_DIR / ".git").exists() else None}
sys.path.insert(0, str(REPO_DIR / "src"))
from neurotutorsim import corpus, tribe

units = corpus.load_units(REPO_DIR / "data" / "units")
stimuli = corpus.load_stimuli(REPO_DIR / "stimuli", units)
corpus_tables = corpus.build_tables(units, stimuli, Path(tempfile.mkdtemp()))
features = pd.DataFrame(corpus_tables["rows"]).set_index(["unit_id", "condition"])
caliper = pd.DataFrame(corpus_tables["caliper"]).T
if not caliper.passed.all():
    raise RuntimeError("duration caliper failed for " + ", ".join(map(str, caliper.index[~caliper.passed])))
ORDER = [u.unit_id for u in corpus.curriculum_order(units)]
print(f"{len(units)} units, {len(stimuli)} stimuli, all {len(caliper)} AI stimuli within the 10 % duration caliper; "
      f"corpus commit {RUN['corpus']['commit']}")
features[["word_count", "sentence_count", "readability", "equation_count", "duration"]].groupby("condition").mean().round(1)

In [ ]:
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path(DRIVE_OUTPUT_DIR)
elif LOCAL_OUTPUT_DIR:
    OUT_ROOT = Path(LOCAL_OUTPUT_DIR)
elif IN_COLAB:
    OUT_ROOT = Path("/content/tribe_outputs")
else:
    OUT_ROOT = REPO_DIR / "data" / "processed" / "tribe"
OUT = OUT_ROOT / (("DRYRUN_" if DRY_RUN else "") + RUN_TAG)
OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path("/content/tribe_cache") if IN_COLAB else OUT_ROOT / "cache"   # text-feature cache, ~13 GB for the corpus
CACHE.mkdir(parents=True, exist_ok=True)
RUN["output_dir"], RUN["cache_dir"] = str(OUT), str(CACHE)
QC = {}


def save_state():
    (OUT / "run_metadata.json").write_text(json.dumps(RUN, indent=2, default=str), encoding="utf-8")
    (OUT / "tribe_qc.json").write_text(json.dumps(QC, indent=2, default=str), encoding="utf-8")


def log_event(**fields):
    with (OUT / "tribe_run_log.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps({"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), **fields}, default=str) + "\n")


for name in ("units.csv", "stimuli.csv"):   # the Phase I index tables travel with the predictions
    shutil.copy(REPO_DIR / "data" / "processed" / name, OUT / name)
save_state()
print("outputs ->", OUT)
if DRY_RUN:
    print("DRY RUN: predictions below are FAKE (seeded noise); use only to check that the pipeline runs.")

## 3. Model checkpoint and verification (§6.1 items 12, 15)

The released checkpoint is downloaded from the Hub and its SHA-256 compared with the LFS object id published
there (no separate checksum file exists). The text encoder is `meta-llama/Llama-3.2-3B` (layers 0.5, 0.75, 1.0,
group-mean), features at 2 Hz, one predicted time point per second on fsaverage5, offset by 5 s to compensate for
the hemodynamic lag: the row at `time_index = t` is the model's estimate of the BOLD response at `t + 5 s`. Only the
data-loader batch size, worker count and encoder precision are changed from the released configuration; every
feature-extraction setting the model was trained with is kept.

In [ ]:
if not DRY_RUN:
    from huggingface_hub import hf_hub_download
    from tribev2 import TribeModel

    ckpt_path = hf_hub_download(TRIBE_HF_REPO, "best.ckpt")
    config_path = hf_hub_download(TRIBE_HF_REPO, "config.yaml")
    ckpt_sha = sha256_of(ckpt_path)
    ckpt_bytes = Path(ckpt_path).stat().st_size
    if ckpt_sha != TRIBE_CKPT_SHA256 or ckpt_bytes != TRIBE_CKPT_BYTES:
        raise RuntimeError(f"checkpoint mismatch: sha256 {ckpt_sha} ({ckpt_bytes} bytes) vs expected {TRIBE_CKPT_SHA256}")
    gpu_gb = RUN.get("gpu", {}).get("memory_gb", 0)
    fp16 = TEXT_PRECISION == "fp16" or (TEXT_PRECISION == "auto" and 0 < gpu_gb < 20)
    config_update = {"data.batch_size": 4, "data.num_workers": 0, "data.text_feature.batch_size": 4,
                     "data.text_feature.device": "accelerate" if fp16 else "cuda"}
    model = TribeModel.from_pretrained(TRIBE_HF_REPO, cache_folder=CACHE, config_update=config_update)
    d = model.data
    RUN["checkpoint"] = {"repo": TRIBE_HF_REPO, "file": "best.ckpt", "sha256": ckpt_sha, "bytes": ckpt_bytes,
                         "config_sha256": sha256_of(config_path), "verified_against_hub_lfs_oid": True}
    RUN["model"] = {
        "tr_s": d.TR, "duration_trs": d.duration_trs, "features_to_use": list(d.features_to_use),
        "text_model": d.text_feature.model_name, "text_layers": d.text_feature.layers,
        "text_cache_n_layers": d.text_feature.cache_n_layers, "feature_hz": d.frequency,
        "hemodynamic_offset_s": d.neuro.offset, "mesh": d.neuro.projection.mesh,
        "modality_dropout_in_training": model.brain_model_config.modality_dropout,
        "text_encoder_precision": "fp16 (accelerate)" if fp16 else "fp32", "config_update": config_update,
        "device": str(model._model.device), "n_parameters": sum(p.numel() for p in model._model.parameters()),
    }
    save_state()
    print(json.dumps(RUN["model"], indent=1, default=str))
else:
    model = None
    print("DRY_RUN: no model loaded")

## 4. Reproduce the official example (§6.1 items 13-14, gate 17)

The official demo notebook predicts responses to a Hamlet passage: text -> Google TTS -> WhisperX word timings ->
TRIBE. On the authors' machine it produced a `(24, 20484)` array (2026-03-30 run); that shape is the summary we
reproduce. If the TTS/WhisperX leg is unavailable in this runtime (both are external services), the same passage is
timed with eq. 4 instead, which still exercises the encoder end to end; which variant ran is recorded in
`tribe_qc.json`, and the prediction itself is kept under `official_demo/`.

In [ ]:
HAMLET = """
To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die, to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
"""


def context_transforms(events):
    """The official text pipeline after word timing: contexts for the encoder, words without one dropped."""
    from neuralset.events.transforms import AddContextToWords, RemoveMissing
    from neuralset.events.utils import standardize_events
    e = standardize_events(events)
    e = AddContextToWords(sentence_only=False, max_context_len=1024, split_field="")(e)
    n_words = int((e.type == "Word").sum())
    e = RemoveMissing()(e)
    return standardize_events(e, auto_fill=False), n_words - int((e.type == "Word").sum())


if RUN_OFFICIAL_DEMO and not DRY_RUN:
    demo_dir = OUT / "official_demo"
    demo_dir.mkdir(exist_ok=True)
    demo = {"expected_shape_official_run": [24, N_VERTICES]}
    text_path = CACHE / "shakespeare.txt"
    text_path.write_text(HAMLET, encoding="utf-8")
    try:
        df_demo = model.get_events_dataframe(text_path=str(text_path))
        demo["timing"] = "official: gTTS + WhisperX word timestamps"
    except Exception as exc:  # noqa: BLE001
        demo["tts_error"] = repr(exc)[:500]
        df_demo, _ = tribe.word_events({"Text": HAMLET}, "hamlet", 220)
        df_demo, _ = context_transforms(df_demo)
        demo["timing"] = "fallback: eq. 4 reading speed 220 wpm (TTS/WhisperX unavailable)"
    t0 = time.time()
    preds_demo, segs_demo = model.predict(events=df_demo)
    demo["shape"] = list(preds_demo.shape)
    demo["seconds"] = round(time.time() - t0, 1)
    demo["finite"] = bool(np.isfinite(preds_demo).all())
    demo["reproduced"] = preds_demo.shape[1] == N_VERTICES and 15 <= preds_demo.shape[0] <= 35 and demo["finite"]
    tribe.write_vertex(demo_dir / "hamlet_vertex.parquet", preds_demo, [int(round(s.start)) for s in segs_demo])
    df_demo.to_parquet(demo_dir / "hamlet_events.parquet")
    QC["official_demo"] = demo
    save_state()
    print(json.dumps(demo, indent=1))
    if not demo["reproduced"]:
        raise RuntimeError("the official example did not reproduce; do not process the corpus (decision gate 17)")
else:
    QC["official_demo"] = {"skipped": True, "reason": "DRY_RUN" if DRY_RUN else "RUN_OFFICIAL_DEMO = False"}
    print("official demo skipped:", QC["official_demo"]["reason"])

## 5. Stimulus timing (§6.2, eq. 4) and inference (§6.3)

Each stimulus is the full markdown body of its file (sections in order, headings excluded), which is the text the
Phase I duration caliper was computed on. A word is one whitespace-delimited token; word `j` starts at
`60 * j / r` seconds and lasts `60 / r` seconds, with `r = 220` wpm in the main specification and 180 / 260 as
robustness arms. Sentence boundaries are preserved: they define the `sentence` field from which the official
`AddContextToWords` transform builds each word's left context for the encoder, exactly as in the released
pipeline. No audio or video is given, so the audio and video pathways receive the zero vector the model was trained
to tolerate (modality dropout 0.3).

TRIBE processes every stimulus in independent 100 s windows (`duration_trs`, the training segment length) and
returns one row per second that overlaps a word; the row's `time_index` is the second since stimulus onset.
Predictions are written per stimulus to `wpm<r>/tribe_vertex/<stimulus_id>.parquet` (compressed columnar, one row
per vertex, one column per second); `tribe.read_vertex` returns the `(T_s, V)` matrix and `tribe.vertex_long`
reshapes it into the brief's long layout on demand.

In [ ]:
def build_events(stimulus_sections, wpm):
    """Events for {stimulus_id: (unit_id, condition, sections)} at reading speed wpm, plus the section table."""
    frames, secs = [], []
    for stimulus_id, (unit_id, condition, sections) in stimulus_sections.items():
        e, s = tribe.word_events(sections, stimulus_id, wpm)
        for frame in (e, s):
            frame["unit_id"], frame["condition"] = unit_id, condition
        frames.append(e)
        secs.append(s)
    events, sections = pd.concat(frames, ignore_index=True), pd.concat(secs, ignore_index=True)
    removed = 0
    if not DRY_RUN:
        events, removed = context_transforms(events)
    return events, sections, removed


def fake_predictions(events):
    """DRY_RUN only: seeded noise with a slow temporal structure, shaped like TRIBE output. Never a result."""
    out = {}
    for timeline, df in events.groupby("timeline", sort=False):
        T = tribe.expected_timepoints(df.stop.max())
        rng = np.random.default_rng([SEED, int(hashlib.sha256(timeline.encode()).hexdigest()[:8], 16)])
        pattern = rng.standard_normal((2, N_VERTICES))
        t = np.arange(T)[:, None]
        preds = (np.sin(2 * np.pi * t / 40) * pattern[0] + np.cos(2 * np.pi * t / 60) * pattern[1]
                 + 0.5 * rng.standard_normal((T, N_VERTICES))).astype(np.float32)
        out[timeline] = (preds, np.arange(T))
    return out


def predict_events(events):
    """TRIBE predictions per timeline: {timeline: (preds (T, V), time_index (T,))}."""
    if DRY_RUN:
        return fake_predictions(events)
    preds, segments = model.predict(events=events, verbose=True)
    rows = {}
    for row, seg in zip(preds, segments):
        rows.setdefault(seg.timeline, []).append((seg.start, row))
    out = {}
    for timeline, items in rows.items():
        items.sort(key=lambda x: x[0])
        out[timeline] = (np.stack([r for _, r in items]), np.array([int(round(s)) for s, _ in items]))
    return out


def run_inference(events, sections, out_dir, label):
    """Predict every timeline not yet on disk, GROUP_SIZE at a time; append-only, resumable."""
    vertex_dir = out_dir / "tribe_vertex"
    vertex_dir.mkdir(parents=True, exist_ok=True)
    timelines = list(dict.fromkeys(events.timeline))
    done = {p.stem for p in vertex_dir.glob("*.parquet")}
    todo = [t for t in timelines if t not in done]
    duration = sections.groupby("timeline").stop.max()
    issues = []
    print(f"[{label}] {len(timelines)} stimuli, {len(done)} already predicted, {len(todo)} to run")
    for i in range(0, len(todo), GROUP_SIZE):
        group = todo[i:i + GROUP_SIZE]
        t0 = time.time()
        results = predict_events(events[events.timeline.isin(group)])
        for timeline in group:
            if timeline not in results:
                issues.append({"timeline": timeline, "problem": "no prediction returned"})
                continue
            preds, time_index = results[timeline]
            expected = tribe.expected_timepoints(duration[timeline])
            if preds.shape != (expected, N_VERTICES) or not np.isfinite(preds).all():
                issues.append({"timeline": timeline, "problem": f"shape {preds.shape} vs expected ({expected}, {N_VERTICES}); "
                                                                f"finite={bool(np.isfinite(preds).all())}"})
            tribe.write_vertex(vertex_dir / f"{timeline}.parquet", preds, time_index)
        elapsed = round(time.time() - t0, 1)
        log_event(stage="inference", label=label, timelines=group, seconds=elapsed)
        print(f"  {i + len(group)}/{len(todo)} done ({elapsed} s for this group)")
    if issues:
        print("ISSUES:", *issues, sep="\n  ")
    return issues


def load_vertex(out_dir, timeline):
    return tribe.read_vertex(out_dir / "tribe_vertex" / f"{timeline}.parquet")


STIMULI = {s.stimulus_id: (uid, cond, s.sections) for (uid, cond), s in stimuli.items()}
EVENTS, SECTIONS = {}, {}
for wpm in READING_SPEEDS:
    out_dir = OUT / f"wpm{wpm}"
    out_dir.mkdir(exist_ok=True)
    events, sections, removed = build_events(STIMULI, wpm)
    events.to_parquet(out_dir / "tribe_events.parquet", index=False)
    sections.to_csv(out_dir / "tribe_sections.csv", index=False)
    words = events[events.type == "Word"]
    per = words.groupby(["unit_id", "condition"]).agg(n_words=("text", "size"), duration_s=("stop", "max"))
    anchor = per.xs("traditional", level="condition").duration_s
    rel = per.reset_index().assign(rel=lambda d: (d.duration_s - d.unit_id.map(anchor)).abs() / d.unit_id.map(anchor))
    QC[f"wpm{wpm}_events"] = {
        "n_stimuli": int(events.timeline.nunique()), "n_words": int(len(words)), "words_without_context_removed": int(removed),
        "stimulus_seconds_total": round(float(per.duration_s.sum()), 1), "duration_s_min": round(float(per.duration_s.min()), 1),
        "duration_s_max": round(float(per.duration_s.max()), 1),
        "token_duration_caliper_max_rel_diff": round(float(rel[rel.condition != "traditional"].rel.max()), 3),
        "expected_timepoints_total": int(sum(tribe.expected_timepoints(d) for d in per.duration_s)),
    }
    EVENTS[wpm], SECTIONS[wpm] = events, sections
    print(f"wpm {wpm}:", json.dumps(QC[f"wpm{wpm}_events"]))
save_state()
EVENTS[READING_SPEEDS[0]].head(8)[["type", "timeline", "start", "duration", "text", "sentence_char", "section"] +
                                  (["context"] if "context" in EVENTS[READING_SPEEDS[0]].columns else [])]

In [ ]:
for wpm in READING_SPEEDS:
    QC[f"wpm{wpm}_inference_issues"] = run_inference(EVENTS[wpm], SECTIONS[wpm], OUT / f"wpm{wpm}", f"wpm{wpm}")
save_state()

## 6. Parcels and networks (§6.4, eq. 6-7)

Vertices are mapped to the Schaefer 2018 parcellation (7-network order) on fsaverage5, taken from the authors'
repository at a pinned commit and checksummed; the network is the third field of each parcel name. Parcel
responses are vertex means (eq. 6); network responses are parcel means weighted by parcel surface area on the
fsaverage5 pial mesh (eq. 7, main) or with equal weights (robustness). Medial-wall vertices belong to no parcel and
are dropped. `parcellation_*.csv` documents the vertex -> parcel -> network mapping.

In [ ]:
import nibabel as nib
from nilearn import datasets

base = f"https://raw.githubusercontent.com/ThomasYeoLab/CBIG/{SCHAEFER_COMMIT}/stable_projects/brain_parcellation/" \
       f"Schaefer2018_LocalGlobal/Parcellations/FreeSurfer5.3/fsaverage5/label/"
annot_dir = OUT / "parcellation"
annot_dir.mkdir(exist_ok=True)
labels, names, area, provenance = {}, {}, {}, {}
fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")
for hemi, fs in (("left", "lh"), ("right", "rh")):
    fname = f"{fs}.Schaefer2018_{SCHAEFER_PARCELS}Parcels_7Networks_order.annot"
    path = annot_dir / fname
    if not path.exists():
        import urllib.request
        urllib.request.urlretrieve(base + fname, path)
    lab, _ctab, nm = nib.freesurfer.read_annot(str(path))
    labels[hemi], names[hemi] = lab, nm
    area_src = fsaverage[f"area_{hemi}"]
    area[hemi] = nib.load(area_src).darrays[0].data if isinstance(area_src, (str, Path)) else np.asarray(area_src)
    provenance[hemi] = {"file": fname, "url": base + fname, "sha256": sha256_of(path)}

parc = tribe.parcellation(labels, names, area)
table = tribe.parcel_table(parc)
if len(parc) != N_VERTICES or table.parcel_id.nunique() != SCHAEFER_PARCELS:
    raise RuntimeError(f"unexpected parcellation: {len(parc)} vertices, {table.parcel_id.nunique()} parcels")
parc.to_csv(OUT / f"parcellation_schaefer{SCHAEFER_PARCELS}_fsaverage5.csv", index=False)
table.to_csv(OUT / f"parcels_schaefer{SCHAEFER_PARCELS}.csv", index=False)
RUN["parcellation"] = {"atlas": f"Schaefer2018 {SCHAEFER_PARCELS} parcels, 7 networks, fsaverage5", "source_commit": SCHAEFER_COMMIT,
                       "files": provenance, "n_vertices": int(len(parc)), "n_medial_wall": int((parc.parcel_id == 0).sum()),
                       "networks": sorted(table.network.unique()), "area_source": "nilearn fsaverage5 area maps (mm^2)"}
save_state()
table.groupby("network").agg(parcels=("parcel_id", "size"), vertices=("n_vertices", "sum"), area_mm2=("area", "sum")).round(0)

## 7. Parcel summaries and immediate neural metrics (§6.5, eq. 8-9)

For every stimulus and every parcel and network: mean over the instructional window, peak and time to peak after
a 3-point moving average, trapezoidal AUC (eq. 8), and sustained engagement (share of seconds above the stimulus's
own median, an explicit assumption). Per stimulus: spatial dispersion (variance of the window-mean parcel pattern),
spatial entropy (eq. 9) and network integration (mean pairwise correlation of the seven network time courses).
Level `network` is area-weighted (main); `network_equal` is the equal-weight robustness check. Representational
differentiation and within-concept consistency are computed offline from `tribe_patterns.parquet`
(`tribe.differentiation`).

In [ ]:
def aggregate_run(wpm):
    """Parcel and network time courses plus the metric table of one reading-speed arm; written to its folder."""
    out_dir = OUT / f"wpm{wpm}"
    parcel_rows, network_rows, metric_rows, patterns = [], [], [], {}
    for stimulus_id, (unit_id, condition, _) in STIMULI.items():
        preds, time_index = load_vertex(out_dir, stimulus_id)
        pmean, psd, pids = tribe.aggregate_parcels(preds, parc)
        net_area, nets = tribe.aggregate_networks(pmean, table, "area")
        net_equal, _ = tribe.aggregate_networks(pmean, table, "equal")
        T, P = pmean.shape
        parcel_rows.append(pd.DataFrame({"stimulus_id": stimulus_id, "unit_id": unit_id, "condition": condition,
                                         "time_index": np.repeat(time_index, P), "parcel_id": np.tile(pids, T),
                                         "mean_bold": pmean.ravel().astype(np.float32), "sd_bold": psd.ravel().astype(np.float32)}))
        network_rows.append(pd.DataFrame({"stimulus_id": stimulus_id, "unit_id": unit_id, "condition": condition,
                                          "time_index": np.repeat(time_index, len(nets)), "network": np.tile(nets, T),
                                          "bold_area": net_area.ravel(), "bold_equal": net_equal.ravel()}))
        m = tribe.stimulus_metrics(pmean, pids, net_area, nets)
        m_equal = tribe.series_metrics(net_equal, nets).assign(level="network_equal")
        metric_rows.append(pd.concat([m, m_equal]).assign(stimulus_id=stimulus_id, unit_id=unit_id, condition=condition))
        patterns[stimulus_id] = pmean.mean(axis=0)
    parcel = pd.concat(parcel_rows, ignore_index=True)
    parcel["network"] = parcel.parcel_id.map(table.set_index("parcel_id").network)
    network = pd.concat(network_rows, ignore_index=True)
    metrics = pd.concat(metric_rows, ignore_index=True)[["stimulus_id", "unit_id", "condition", "level", "key", "metric", "value"]]
    parcel.to_parquet(out_dir / "tribe_parcel.parquet", index=False)
    network.to_parquet(out_dir / "tribe_network.parquet", index=False)
    metrics.to_parquet(out_dir / "tribe_metrics.parquet", index=False)
    wide = metrics[metrics.level.isin(["network", "stimulus"])].pivot_table(
        index=["unit_id", "condition"], columns=["key", "metric"], values="value")
    wide.columns = [f"{k}_{m}" for k, m in wide.columns]
    wide.to_csv(out_dir / "tribe_metrics_network.csv")
    patterns = pd.DataFrame(patterns, index=pids).T  # rows stimulus_id, columns parcel_id: the z_uc of §6.7
    patterns.to_parquet(out_dir / "tribe_patterns.parquet")
    return parcel, network, metrics, patterns


RESULTS = {}
for wpm in READING_SPEEDS:
    RESULTS[wpm] = aggregate_run(wpm)
    print(f"wpm {wpm}: parcel table {RESULTS[wpm][0].shape}, metrics table {RESULTS[wpm][2].shape}")
main_metrics = RESULTS[READING_SPEEDS[0]][2]
main_metrics[main_metrics.level == "network"].pivot_table(index="key", columns=["metric", "condition"], values="value").round(3)

## 8. Validation controls (§10.1)

* **Determinism.** One stimulus is predicted again by a second model instance with an empty feature cache; the
  maximum absolute difference is reported (fp16 encoders can differ in the last digit).
* **Shuffled text.** For `N_CONTROL_UNITS` units x 3 conditions, the sentence order and the word order are
  permuted (same words, same duration) and predicted; the network metrics of the shuffled stimuli are saved next to
  the intact ones so the change they induce can be compared offline with the condition contrasts.
* Held-out naturalistic fMRI validation (§10.1, second bullet) is **not** done here: it needs licensed fMRI data.

In [ ]:
controls = {}
main_wpm = READING_SPEEDS[0]
main_dir = OUT / f"wpm{main_wpm}"
if RUN_CONTROLS:
    ctrl_dir = OUT / "controls"
    ctrl_dir.mkdir(exist_ok=True)
    first = next(iter(STIMULI))
    ev_first = EVENTS[main_wpm][EVENTS[main_wpm].timeline == first]
    if DRY_RUN:
        twice = [fake_predictions(ev_first)[first][0] for _ in range(2)]
    else:
        cache2 = CACHE.parent / (CACHE.name + "_determinism")
        shutil.rmtree(cache2, ignore_errors=True)
        model2 = TribeModel.from_pretrained(TRIBE_HF_REPO, cache_folder=cache2, config_update=config_update)
        twice = [load_vertex(main_dir, first)[0]]
        preds2, segs2 = model2.predict(events=ev_first, verbose=False)
        twice.append(np.stack([r for _, r in sorted(zip([s.start for s in segs2], preds2), key=lambda x: x[0])]))
        del model2
    diff = float(np.abs(twice[0] - twice[1]).max()) if twice[0].shape == twice[1].shape else float("nan")
    controls["determinism"] = {"stimulus": first, "shapes": [list(t.shape) for t in twice], "max_abs_diff": diff,
                               "identical": bool(diff == 0), "within_1e-3": bool(diff < 1e-3)}
    print("determinism:", controls["determinism"])

    rng = np.random.default_rng(SEED)
    ctrl_stimuli = {}
    for uid in ORDER[:N_CONTROL_UNITS]:
        for cond in corpus.CONDITIONS:
            sid = f"{uid}_{cond}"
            for how in ("sentence", "word"):
                ctrl_stimuli[f"{sid}__shuffled_{how}"] = (uid, cond, tribe.shuffled_sections(STIMULI[sid][2], how, rng))
    ev_ctrl, sec_ctrl, _ = build_events(ctrl_stimuli, main_wpm)
    ev_ctrl.to_parquet(ctrl_dir / "tribe_events_controls.parquet", index=False)
    controls["shuffled_inference_issues"] = run_inference(ev_ctrl, sec_ctrl, ctrl_dir, "controls")
    rows = []
    for sid, (uid, cond, _) in ctrl_stimuli.items():
        preds, _ = load_vertex(ctrl_dir, sid)
        pmean, _, pids = tribe.aggregate_parcels(preds, parc)
        net, nets = tribe.aggregate_networks(pmean, table, "area")
        rows.append(tribe.series_metrics(net, nets).assign(stimulus_id=sid, unit_id=uid, condition=cond,
                                                            control=sid.split("__shuffled_")[1]))
    ctrl_metrics = pd.concat(rows, ignore_index=True)
    intact = main_metrics[main_metrics.level == "network"].rename(columns={"value": "intact"})
    ctrl_metrics = ctrl_metrics.merge(intact[["unit_id", "condition", "key", "metric", "intact"]],
                                      on=["unit_id", "condition", "key", "metric"])
    ctrl_metrics["abs_change"] = (ctrl_metrics.value - ctrl_metrics.intact).abs()
    ctrl_metrics.to_csv(ctrl_dir / "tribe_controls_shuffled.csv", index=False)
    controls["shuffled_mean_abs_change"] = ctrl_metrics.groupby(["control", "metric"]).abs_change.mean().round(4).unstack("control").to_dict()
    display(ctrl_metrics.groupby(["control", "metric"]).abs_change.mean().unstack("control").round(4))
QC["controls"] = controls if RUN_CONTROLS else {"skipped": True}
save_state()

## 9. Run record and download

In [ ]:
RUN["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
RUN["outputs"] = sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file() and "tribe_vertex" not in p.parts)
RUN["vertex_files"] = {f"wpm{wpm}": len(list((OUT / f"wpm{wpm}" / "tribe_vertex").glob("*.parquet"))) for wpm in READING_SPEEDS}
save_state()
print(json.dumps({"vertex_files": RUN["vertex_files"], "issues": {k: v for k, v in QC.items() if k.endswith("_issues")}}, indent=1))

zip_path = OUT_ROOT / f"{OUT.name}_summary.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and (ZIP_INCLUDES_VERTEX or "tribe_vertex" not in p.parts):
            z.write(p, p.relative_to(OUT.parent))
print(f"summary zip: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB); vertex predictions stay in {OUT}/wpm*/tribe_vertex/")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

## What is where

| File (per `wpm<r>/` unless noted) | One row per | Fields |
|---|---|---|
| `tribe_vertex/<stimulus_id>.parquet` | vertex | `vertex_id`, one column per second `t000000..` of predicted BOLD (z); `tribe.read_vertex` returns (T, V), `tribe.vertex_long` the brief's `(stimulus_id, time_index, vertex_id, predicted_bold)` layout |
| `tribe_parcel.parquet` | stimulus x second x parcel | `stimulus_id, unit_id, condition, time_index, parcel_id, network, mean_bold, sd_bold` (brief §4.2, eq. 6) |
| `tribe_network.parquet` | stimulus x second x network | `bold_area` (eq. 7, main), `bold_equal` (robustness) |
| `tribe_metrics.parquet` | stimulus x level x key x metric | §6.5 metrics; `level` in parcel / network / network_equal / stimulus |
| `tribe_metrics_network.csv` | stimulus | the network and stimulus-level metrics, wide |
| `tribe_patterns.parquet` | stimulus | window-mean parcel pattern (the `z_uc` of §6.7) |
| `tribe_events.parquet`, `tribe_sections.csv` | word / section | the exact model input with contexts; section onsets in seconds |
| `parcellation_*.csv`, `parcels_*.csv` (top level) | vertex / parcel | the documented vertex -> parcel -> network mapping with areas |
| `official_demo/`, `controls/` (top level) | | the §6.1 reproduction and the §10.1 controls |
| `run_metadata.json`, `tribe_qc.json`, `tribe_run_log.jsonl` (top level) | | checkpoint hash, commits, hardware, precision, timings, every check |

**Phase IV join.** `tribe_metrics.parquet` filtered to `level == "parcel"`, `metric == "auc"` is `AUC_u,c,p` of
eq. 28, keyed by `(unit_id, condition)` like `learner_state.parquet` in the simulation.

**Offline, no GPU.** `tribe.paired_contrasts(metrics)` and `tribe.fixed_effects(metrics)` give eq. 10-13 (Table 4);
`tribe.rsa({...})` and `tribe.differentiation(...)` on `tribe_patterns.parquet` give §6.7.

**Not done here.** The TTS/audio stimulus arm (§5.2 item 10) and validation against held-out naturalistic fMRI
(§10.1) are left out: the first is optional in the brief and doubles the inference cost, the second needs licensed
data.